# Music Annotation Project — Notebook 38
## V8.2i Envelope Relaxation and Perceptual Severity Calibration — Amendment 38.2

**Status:** development-only, **single-reviewer exploratory**. CPU is sufficient. Run in order.

This notebook does not retrain, refit or recalibrate V8.2; reinterpret truth; change Notebook 37 decisions; use the untouched future cohort; adopt a threshold; freeze a challenger; or authorise production.

### Amendment 38.2
`upload/upload_manifest.csv` is treated as a manifest of Notebook 37's curated upload package, not as the authority for same-named files in the Notebook 37 run root. Every run-root file directly consumed here must exist and is independently content-addressed. Package-copy discrepancies remain explicit provenance warnings and qualify the final exploratory verdict; they are not mislabelled as corruption of the run-root analytical tables.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q "gradio>=4.44,<6" pandas numpy pyarrow matplotlib pillow scipy


## 1. Configuration


In [ ]:
from pathlib import Path
from datetime import datetime,timezone
import base64,hashlib,itertools,json,math,os,shutil,zipfile
import numpy as np,pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
NB37_ROOT=ROOT/'observed_print_classifier_v8_2h_threshold_frontier'
OUTPUT_ROOT=ROOT/'observed_print_classifier_v8_2i_envelope_relaxation'
SOURCE_NB37_RUN_OVERRIDE=None
NOTEBOOK_VERSION='38-v8.2i-amended-38.2';PROTOCOL_VERSION='1.0-single-reviewer-exploratory';REVIEWER_ID='reviewer_01'
ASSIGNMENT_SEED=38001;BOOTSTRAP_SEED=37001;BOOTSTRAP_RESAMPLES=10_000;CALIBRATION_FRACTION=.50
THRESHOLDS=[.995,.990,.985,.980,.975,.950,.925,.900,.875,.850,.825,.800,.750,.700,.600,.500]
FP_WEIGHTS=[1,2,3,5,10];P1_WEIGHTS=[0,1];P2_WEIGHTS=[5,10,25];P3_WEIGHTS=[100,250,500]
D7_TP=[100,250,500,750,1000];D7_PAGES=[5,8,10,15];D7_GROUPS=[2,3,4,5]
RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_v8_2i_envelope_relaxation';OUT=OUTPUT_ROOT/RUN;TABLES=OUT/'tables';REVIEW=OUT/'review';AUDIT=OUT/'audit';VIS=OUT/'visuals';REF=OUT/'reference';UPLOAD=OUT/'upload'
for p in [OUT,TABLES,REVIEW,AUDIT,VIS,REF]:p.mkdir(parents=True,exist_ok=False)
print('Output:',OUT)


## 2. Helpers and protocol lock


In [ ]:
def now():return datetime.now(timezone.utc).isoformat()
def sha(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''):h.update(b)
 return h.hexdigest()
def text(p,s):
 p=Path(p);p.parent.mkdir(parents=True,exist_ok=True);q=p.with_name(p.name+'.tmp');q.write_text(s,encoding='utf-8');os.replace(q,p)
def js(p,x):text(p,json.dumps(x,indent=2,sort_keys=True,default=str))
def csv(d,p):
 p=Path(p);p.parent.mkdir(parents=True,exist_ok=True);q=p.with_name(p.name+'.tmp');d.to_csv(q,index=False);os.replace(q,p)
def stop(stage,msg,details=None):js(AUDIT/f'failure_{stage}.json',{'stage':stage,'message':msg,'details':details or {},'utc':now()});raise RuntimeError(f'{stage}: {msg}')
def col(d,names,required=True):
 z={str(x).lower():x for x in d.columns}
 for n in names:
  if n.lower() in z:return z[n.lower()]
 if required:raise KeyError(f'Missing one of {names}; columns={list(d.columns)}')
def okimg(p):
 try:
  with Image.open(p) as im:im.verify()
  return True
 except:return False
def rec_hash(d):return hashlib.sha256(json.dumps(d,sort_keys=True,default=str).encode()).hexdigest()
LOCK={'notebook_version':NOTEBOOK_VERSION,'protocol_version':PROTOCOL_VERSION,'created_utc':now(),'status':'development-only-single-reviewer-exploratory','reviewer_count':1,'thresholds':THRESHOLDS,'assignment_seed':ASSIGNMENT_SEED,'bootstrap_seed':BOOTSTRAP_SEED,'registered_utility':'TP-5*FP-500*C','may_nominate_revised_envelope':False,'production_authority':False,'untouched_future_validation_used':False,'sparse_hard_negative':{'role':'explicitly sparse training-role material','D9':'not validated'}}
js(OUT/'protocol_lock.json',LOCK)


## 3. Resolve and lock direct Notebook 37 source inputs


In [ ]:
REQUIRED=['experiment_identity.json','source_v82_record.json','executive_summary.txt','tables/threshold_frontier_summary.csv','tables/threshold_frontier_by_page.csv','tables/threshold_frontier_by_source_group.csv','tables/incremental_threshold_band_summary.csv','tables/incremental_threshold_pixels.parquet','tables/incremental_component_review_queue_all.csv','tables/catastrophic_components_all_thresholds.csv','tables/catastrophic_component_first_appearance_inventory.csv','tables/computable_gate_map.csv','tables/representative_page_selection.csv','audit/whole_page_integrity_and_hashes.csv','upload/upload_manifest.csv']
def runs(root):
 if not Path(root).exists():return []
 return sorted([p for p in Path(root).iterdir() if p.is_dir() and all((p/r).exists() for r in REQUIRED[:8])],key=lambda p:p.stat().st_mtime)
NB37=Path(SOURCE_NB37_RUN_OVERRIDE) if SOURCE_NB37_RUN_OVERRIDE else (runs(NB37_ROOT)[-1] if runs(NB37_ROOT) else None)
if NB37 is None:stop('input_resolution','No complete-looking Notebook 37 run found',{'root':str(NB37_ROOT)})
rows=[]
for r in REQUIRED:
 p=NB37/r;rows.append({'relative_path':r,'absolute_path':str(p.resolve()) if p.exists() else '','exists':p.exists(),'bytes':p.stat().st_size if p.is_file() else np.nan,'sha256':sha(p) if p.is_file() else ''})
source_lock=pd.DataFrame(rows);csv(source_lock,AUDIT/'notebook38_source_input_lock.csv')
missing=source_lock.loc[~source_lock.exists,'relative_path'].tolist()
if missing:stop('input_resolution','Mandatory direct Notebook 37 inputs are missing',{'missing':missing})
source_lock_sha256=hashlib.sha256(source_lock.to_csv(index=False,lineterminator='\n').encode()).hexdigest();LOCK.update({'source_nb37_run':str(NB37),'direct_source_lock_sha256':source_lock_sha256});js(OUT/'protocol_lock.json',LOCK)
print('Direct source lock:',source_lock_sha256)


## 4. Package-scoped Notebook 37 upload-manifest audit


In [ ]:
manifest=pd.read_csv(NB37/'upload/upload_manifest.csv');pc=col(manifest,['relative_path','path','file','filename']);hc=col(manifest,['sha256','sha_256','hash'],False)
def package_candidates(raw):
 s=str(raw).strip().replace('\\','/');q=Path(s);out=[]
 if q.is_absolute():out.append(q)
 parts=list(q.parts)
 if 'upload' in parts:
  k=parts.index('upload');tail=Path(*parts[k+1:]) if k+1<len(parts) else Path();out.append(NB37/'upload'/tail)
 out.extend([NB37/'upload'/q,NB37/q]);hits=[x for x in (NB37/'upload').rglob(q.name) if x.is_file()] if q.name else []
 if len({str(x.resolve()) for x in hits})==1:out.extend(hits)
 ans=[];seen=set()
 for x in out:
  if str(x) not in seen:ans.append(x);seen.add(str(x))
 return ans
checks=[]
for _,r in manifest.iterrows():
 raw=str(r[pc]);p=next((x for x in package_candidates(raw) if x.is_file()),None);actual=sha(p) if p else '';expected=str(r[hc]).strip() if hc and pd.notna(r[hc]) else '';match=(actual.lower()==expected.lower()) if expected and p else np.nan;status='verified_package_copy' if p and (not expected or match is True) else ('package_copy_hash_mismatch' if p and match is False else 'unresolved_package_row');checks.append({'manifest_path':raw,'resolved_package_path':str(p or ''),'exists':bool(p),'expected_sha256':expected,'actual_sha256':actual,'hash_matches':match,'status':status})
package_audit=pd.DataFrame(checks);csv(package_audit,AUDIT/'nb37_manifest_verification.csv');warning_count=int(package_audit.status.ne('verified_package_copy').sum());manifest_summary={'scope':'Notebook 37 curated upload package only','rows':len(package_audit),'verified_package_copies':int(package_audit.status.eq('verified_package_copy').sum()),'unresolved_package_rows':int(package_audit.status.eq('unresolved_package_row').sum()),'package_copy_hash_mismatches':int(package_audit.status.eq('package_copy_hash_mismatch').sum()),'direct_source_files_locked':len(source_lock),'direct_source_lock_sha256':source_lock_sha256,'provenance_warning':bool(warning_count),'interpretation':'Package discrepancies do not assert corruption of separately fingerprinted run-root analytical inputs.'};js(AUDIT/'nb37_manifest_verification_summary.json',manifest_summary);LOCK.update({'notebook37_package_manifest_summary':manifest_summary,'result_qualification_required':bool(warning_count)});js(OUT/'protocol_lock.json',LOCK);print(json.dumps(manifest_summary,indent=2))


## 5. Threshold identity and registered D3–D8 replay


In [ ]:
frontier=pd.read_csv(NB37/'tables/threshold_frontier_summary.csv');actual=[round(float(x),3) for x in frontier[col(frontier,['threshold'])]]
if actual!=THRESHOLDS:stop('threshold_identity','Threshold identity/order mismatch',{'expected':THRESHOLDS,'actual':actual})
need=['threshold','true_print','false_annotation','truth_uncertain_print','selected_print_pixels','utility','catastrophic_components','false_true_ratio','false_preservation_rate','wilson95_upper_false_preservation','bootstrap_utility_95_low','bootstrap_false_true_95_high','benefiting_pages','benefiting_source_groups','source_stability_for_groups_with_3_pages','D3_catastrophic_safety','D4_false_preservation','D5_pixel_safety','D6_positive_utility','D7_nontrivial_benefit','D8_source_stability','core_development_envelope_D3_to_D8']
miss=[x for x in need if x not in frontier]
if miss:stop('registered_replay','Missing registered fields',{'columns':miss})
rep=pd.DataFrame({'threshold':frontier.threshold,'D3':frontier.catastrophic_components.eq(0),'D4':frontier.false_true_ratio.le(.10)&frontier.bootstrap_false_true_95_high.le(.20),'D5':frontier.false_preservation_rate.le(.005)&frontier.wilson95_upper_false_preservation.le(.01),'D6':frontier.utility.gt(0)&frontier.bootstrap_utility_95_low.gt(0),'D7':frontier.true_print.ge(500)&frontier.benefiting_pages.ge(10)&frontier.benefiting_source_groups.ge(3),'D8':frontier.source_stability_for_groups_with_3_pages.astype(bool)})
for i,s in enumerate(['D3_catastrophic_safety','D4_false_preservation','D5_pixel_safety','D6_positive_utility','D7_nontrivial_benefit','D8_source_stability'],3):
 if not np.array_equal(rep[f'D{i}'].to_numpy(),frontier[s].astype(bool).to_numpy()):stop('registered_replay',f'D{i} mismatch')
csv(frontier[need],TABLES/'registered_D3_D8_results.csv');csv(rep,AUDIT/'registered_gate_replay.csv');shutil.copy2(NB37/'experiment_identity.json',REF/'notebook37_experiment_identity.json');shutil.copy2(NB37/'source_v82_record.json',REF/'source_v82_record.json');print('Registered replay verified')


## 6. Stable threshold-transition review units and group split


In [ ]:
q=pd.read_csv(NB37/'tables/incremental_component_review_queue_all.csv');cf=pd.read_csv(NB37/'tables/catastrophic_component_first_appearance_inventory.csv');aliases={'page':['page'],'source_group':['source_group'],'mask_component_id':['mask_component_id','component_id'],'higher_threshold':['higher_threshold','from_threshold','high_threshold'],'lower_threshold':['lower_threshold','to_threshold','low_threshold'],'new_selected':['new_selected','new_selected_pixels','incremental_selected','selected'],'new_true_print':['new_true_print','incremental_true_print','true_print'],'new_false_annotation':['new_false_annotation','incremental_false_annotation','false_annotation'],'new_truth_uncertain':['new_truth_uncertain','incremental_truth_uncertain','truth_uncertain'],'local_panel':['local_panel','local_crop','local_path'],'wide_panel':['wide_panel','wide_crop','wide_path'],'locator':['locator','whole_page_locator','locator_path'],'truth_overlay':['truth_overlay','cumulative_truth_overlay','incremental_truth_overlay','truth_overlay_path']};u=pd.DataFrame()
for target,names in aliases.items():
 x=col(q,names,target in ['page','source_group','mask_component_id','higher_threshold','lower_threshold'])
 if x is not None:u[target]=q[x]
for x in ['new_selected','new_true_print','new_false_annotation','new_truth_uncertain']:
 if x not in u:u[x]=0
 u[x]=pd.to_numeric(u[x],errors='coerce').fillna(0).astype(int)
u.page=u.page.astype(str);u.source_group=u.source_group.astype(str);u.mask_component_id=pd.to_numeric(u.mask_component_id).astype(int);u.higher_threshold=pd.to_numeric(u.higher_threshold).round(3);u.lower_threshold=pd.to_numeric(u.lower_threshold).round(3)
a=cf[[col(cf,['page']),col(cf,['mask_component_id','component_id']),col(cf,['first_appearance_threshold','threshold','lower_threshold'])]].copy();a.columns=['page','mask_component_id','first_appearance_threshold'];a.page=a.page.astype(str);a.mask_component_id=pd.to_numeric(a.mask_component_id).astype(int);a['registered_catastrophic']=True;u=u.merge(a,on=['page','mask_component_id'],how='left');u.registered_catastrophic=u.registered_catastrophic.fillna(False).astype(bool);u['lineage_key']=u.apply(lambda r:f'{r.page}|{r.source_group}|{r.mask_component_id}|{r.higher_threshold:.3f}|{r.lower_threshold:.3f}',axis=1);u['review_unit_id']=u.lineage_key.map(lambda x:'RU_'+hashlib.sha256(f'{ASSIGNMENT_SEED}|{x}'.encode()).hexdigest()[:16])
if u.review_unit_id.duplicated().any():stop('review_units','Duplicate units')
def asset(v):
 if pd.isna(v) or not str(v).strip():return ''
 p=Path(str(v))
 for z in [p,NB37/p,NB37/'review'/p,NB37/'visual_audit'/p]:
  if z.is_file():return str(z)
 return ''
for x in ['local_panel','wide_panel','locator','truth_overlay']:
 if x not in u:u[x]=''
 else:u[x]=u[x].map(asset)
g=u.groupby('source_group',as_index=False).agg(review_units=('review_unit_id','size'),catastrophic_units=('registered_catastrophic','sum'),pages=('page','nunique'));rng=np.random.default_rng(ASSIGNMENT_SEED);g['_tie']=rng.random(len(g));g=g.sort_values(['catastrophic_units','review_units','_tie'],ascending=[False,False,True]);target=round(len(u)*CALIBRATION_FRACTION);ca=[];au=[];n=0
for _,r in g.iterrows():
 if n<target:ca.append(r.source_group);n+=int(r.review_units)
 else:au.append(r.source_group)
if len(g)>=2 and (not ca or not au):ca=[g.iloc[0].source_group];au=g.iloc[1:].source_group.tolist()
g['assignment_split']=np.where(g.source_group.isin(ca),'rubric_calibration','audit_reserve');g=g.drop(columns='_tie');u=u.merge(g[['source_group','assignment_split']],on='source_group',validate='many_to_one');csv(g,TABLES/'review_split_map.csv');csv(u,TABLES/'review_unit_manifest.csv');print('Units',len(u),'groups',len(g))


## 7. Resolve and blind whole-page A/B pairs


In [ ]:
def toks(t):return [f'{t:.3f}',f'{t:.3f}'.replace('.',''),f'{t:.3f}'.replace('.','_'),str(float(t))]
def whole(page,t):
 d=NB37/'whole_pages'/str(page);scores=[]
 if not d.exists():return None
 for p in d.rglob('*'):
  if p.suffix.lower() not in ['.png','.jpg','.jpeg']:continue
  s=sum(x in p.name.lower() for x in toks(t))-3*sum(x in p.name.lower() for x in ['difference','overlay','decision','locator'])
  if s>0:scores.append((s,p))
 return max(scores,key=lambda z:z[0])[1] if scores else None
def observed(page):
 d=NB37/'whole_pages'/str(page)
 for pat in ['observed.png','*observed*.png','*annotated*.png']:
  z=list(d.glob(pat)) if d.exists() else []
  if z:return str(z[0])
 return ''
rows=[];missing=[]
for _,r in u.iterrows():
 hi=whole(r.page,r.higher_threshold);lo=whole(r.page,r.lower_threshold)
 if not hi or not lo or not okimg(hi) or not okimg(lo):missing.append({'review_unit_id':r.review_unit_id,'page':r.page});continue
 left=int(hashlib.sha256(f'{ASSIGNMENT_SEED}|{r.review_unit_id}'.encode()).hexdigest(),16)%2==0;A,B=(lo,hi) if left else (hi,lo);mapping={'A':'lower','B':'higher'} if left else {'A':'higher','B':'lower'};rows.append({**r.to_dict(),'whole_page_A':str(A),'whole_page_B':str(B),'A_sha256':sha(A),'B_sha256':sha(B),'observed_path':observed(r.page),'randomisation_side_locked':base64.urlsafe_b64encode(json.dumps(mapping).encode()).decode()})
csv(pd.DataFrame(missing),AUDIT/'missing_review_assets.csv')
if missing:stop('review_assets','Missing/unreadable whole-page pairs',{'count':len(missing),'examples':missing[:10]})
blind=pd.DataFrame(rows);csv(blind,TABLES/'blinded_review_manifest.csv');print('Blinded units',len(blind))


## 8. Gradio review — first pass locked before diagnostics


In [ ]:
import gradio as gr
R1=REVIEW/'perceptual_review_round1.csv';R2=REVIEW/'diagnostic_review_round2.csv';R1C=['review_unit_id','reviewer_id','initial_visibility','pair_preference','harm_severity','benefit_severity','defect_scope','musical_function_at_risk','annotation_prominence','reviewer_action','confidence','review_note','timestamp_utc','locked_before_diagnostic','record_sha256'];R2C=['review_unit_id','reviewer_id','diagnostic_mechanism','resolved_harm_severity','resolved_benefit_severity','diagnostic_note','timestamp_utc','record_sha256']
def records(p,cols):
 if not Path(p).exists():return pd.DataFrame(columns=cols)
 d=pd.read_csv(p,keep_default_na=False)
 for x in cols:
  if x not in d:d[x]=''
 return d[cols]
def save(p,cols,d):
 d['record_sha256']=rec_hash(d);z=records(p,cols);z=z[z.review_unit_id.astype(str)!=str(d['review_unit_id'])];csv(pd.concat([z,pd.DataFrame([d])],ignore_index=True)[cols],p);return d['record_sha256']
def pending():
 done=set(records(R2,R2C).review_unit_id.astype(str));return [i for i,x in enumerate(blind.review_unit_id.astype(str)) if x not in done]
def img(x):return str(x) if str(x) and Path(str(x)).is_file() else None
def view(i):
 i=int(max(0,min(i,len(blind)-1)));r=blind.iloc[i];return i,img(r.whole_page_A),img(r.whole_page_B),f'Item {i+1}/{len(blind)} — complete {len(records(R2,R2C))}/{len(blind)}',r.review_unit_id
def first(i,vis,pref,harm,benefit,scope,musical,prom,action,conf,note):
 r=blind.iloc[int(i)];vals=[vis,pref,harm,benefit,scope,musical,prom,action,conf]
 if any(x in [None,''] for x in vals):raise gr.Error('Complete all fields.')
 d={'review_unit_id':r.review_unit_id,'reviewer_id':REVIEWER_ID,'initial_visibility':vis,'pair_preference':pref,'harm_severity':harm,'benefit_severity':benefit,'defect_scope':scope,'musical_function_at_risk':musical,'annotation_prominence':prom,'reviewer_action':action,'confidence':conf,'review_note':str(note),'timestamp_utc':now(),'locked_before_diagnostic':True};h=save(R1,R1C,d);return f'Locked {h[:12]}',gr.update(visible=False),gr.update(visible=True),img(r.wide_panel),img(r.local_panel),img(r.locator),img(r.observed_path),img(r.truth_overlay)
def diag(i,mech,harm,benefit,note):
 r=blind.iloc[int(i)]
 if not mech or not harm or not benefit:raise gr.Error('Complete diagnostic fields.')
 d={'review_unit_id':r.review_unit_id,'reviewer_id':REVIEWER_ID,'diagnostic_mechanism':mech,'resolved_harm_severity':harm,'resolved_benefit_severity':benefit,'diagnostic_note':str(note),'timestamp_utc':now()};h=save(R2,R2C,d);p=pending();nxt=next((x for x in p if x>int(i)),p[0] if p else int(i));v=view(nxt);return *v,f'Saved {h[:12]}',gr.update(visible=True),gr.update(visible=False)
with gr.Blocks(title='Notebook 38 Review') as app:
 gr.Markdown('# Blinded whole-page review');state=gr.State(pending()[0] if pending() else 0);unit=gr.Textbox(label='Opaque unit');status=gr.Markdown()
 with gr.Row():A=gr.Image(label='A',type='filepath',height=700);B=gr.Image(label='B',type='filepath',height=700)
 with gr.Column(visible=True) as p1:
  vis=gr.Radio(['no difference','visible difference','unclear'],label='Visibility');pref=gr.Radio(['A','B','neither','mixed','unclear'],label='Preference');harm=gr.Radio(['P0','P1','P2','P3','P4','U'],label='Harm');benefit=gr.Radio(['B0','B1','B2','B3','B4','U'],label='Benefit');scope=gr.Dropdown(['pixel-fragment','component-local','staff/system','page-repeated','page-global'],label='Scope');musical=gr.Dropdown(['none','staff','notehead','stem/beam','accidental','rest','articulation','dynamic','text/lyric','barline/repeat','other','unclear'],label='Musical risk');prom=gr.Radio(['none','faint','moderate','prominent'],label='Prominence');action=gr.Radio(['accept','accept-with-note','manual-correct','reject-policy-case','unclear'],label='Action');conf=gr.Radio(['low','medium','high'],label='Confidence');note1=gr.Textbox(label='Note');b1=gr.Button('Lock first pass',variant='primary')
 with gr.Column(visible=False) as p2:
  with gr.Row():wide=gr.Image(label='Wide');local=gr.Image(label='Local');locator=gr.Image(label='Locator');obs=gr.Image(label='Observed');truth=gr.Image(label='Truth/overlay')
  mech=gr.Dropdown(['correct print recovery','retained annotation','mixed overlap','registration ambiguity','truth-boundary sensitivity','display artefact','other'],label='Mechanism');rh=gr.Radio(['P0','P1','P2','P3','P4','U'],label='Resolved harm');rb=gr.Radio(['B0','B1','B2','B3','B4','U'],label='Resolved benefit');note2=gr.Textbox(label='Diagnostic note');b2=gr.Button('Save and next',variant='primary')
 msg=gr.Markdown();app.load(view,[state],[state,A,B,status,unit]);b1.click(first,[state,vis,pref,harm,benefit,scope,musical,prom,action,conf,note1],[msg,p1,p2,wide,local,locator,obs,truth]);b2.click(diag,[state,mech,rh,rb,note2],[state,A,B,status,unit,msg,p1,p2])
app.launch(share=True,debug=False)


## 9. Completion, analysis and verdict


In [ ]:
r1=records(R1,R1C);r2=records(R2,R2C);ids=set(blind.review_unit_id.astype(str));coverage=pd.DataFrame({'review_unit_id':sorted(ids),'round1_complete':[x in set(r1.review_unit_id.astype(str)) for x in sorted(ids)],'round2_complete':[x in set(r2.review_unit_id.astype(str)) for x in sorted(ids)]});csv(coverage,AUDIT/'review_coverage.csv')
if not coverage[['round1_complete','round2_complete']].all().all():stop('review_completion','Review incomplete')
joined=blind.merge(r1,on='review_unit_id',validate='one_to_one').merge(r2,on=['review_unit_id','reviewer_id'],validate='one_to_one');hn={'P0':0,'P1':1,'P2':2,'P3':3,'P4':4};bn={'B0':0,'B1':1,'B2':2,'B3':3,'B4':4};joined['harm_n']=joined.resolved_harm_severity.map(hn);joined['benefit_n']=joined.resolved_benefit_severity.map(bn)
if joined[['harm_n','benefit_n']].isna().any().any():stop('analysis','Unresolved U remains')
cal=joined[joined.assignment_split.eq('rubric_calibration')];audit=joined[joined.assignment_split.eq('audit_reserve')];sev={'protocol_version':PROTOCOL_VERSION,'calibration_groups':sorted(cal.source_group.unique()),'weights':{'FP':FP_WEIGHTS,'P1':P1_WEIGHTS,'P2':P2_WEIGHTS,'P3':P3_WEIGHTS,'P4':'hard_fail'},'single_reviewer_exploratory':True};js(OUT/'severity_mapping_lock.json',sev);csv(pd.DataFrame([{'reviewer_count':1,'status':'not_estimable_single_reviewer'}]),TABLES/'reviewer_agreement.csv')
pg=pd.read_csv(NB37/'tables/threshold_frontier_by_page.csv');sg=pd.read_csv(NB37/'tables/threshold_frontier_by_source_group.csv');ag=set(g.loc[g.assignment_split.eq('audit_reserve'),'source_group'].astype(str));pg=pg[pg.source_group.astype(str).isin(ag)];sg=sg[sg.source_group.astype(str).isin(ag)]
def active(t):return audit[audit.lower_threshold.astype(float).ge(t)]
P=[];D8=[];U=[];D7=[]
for t in THRESHOLDS:
 p=pg[np.isclose(pg.threshold,t)];s=sg[np.isclose(sg.threshold,t)];r=active(t)
 if p.empty:continue
 tp=int(p.true_print.sum());fp=int(p.false_annotation.sum());cats=int(p.catastrophic_components.sum());sel=int(p.selected.sum());p2=r.harm_n.ge(2);p3=r.harm_n.ge(3);p4=r.harm_n.eq(4);rep=bool((r.loc[p3].groupby(['source_group','diagnostic_mechanism']).size()>=2).any()) if p3.any() else False;P.append({'threshold':t,'true_print':tp,'false_annotation':fp,'catastrophes':cats,'utility':tp-5*fp-500*cats,'false_true':fp/max(tp,1),'false_preservation':fp/max(sel,1),'P2_plus':int(p2.sum()),'P3_plus':int(p3.sum()),'P4':int(p4.sum()),'repeated_P3':rep,'D3_R0':cats==0,'D3_R1':not p2.any(),'D3_R2':not p3.any(),'D3_R3':not p3.any()})
 counts=r.resolved_harm_severity.value_counts().to_dict()
 for w,a,b,d in itertools.product(FP_WEIGHTS,P1_WEIGHTS,P2_WEIGHTS,P3_WEIGHTS):U.append({'threshold':t,'FP_weight':w,'P1_weight':a,'P2_weight':b,'P3_weight':d,'P4_hard_fail':bool(p4.any()),'severity_adjusted_utility':tp-w*fp-counts.get('P1',0)*a-counts.get('P2',0)*b-counts.get('P3',0)*d})
 for label,minb in [('B1+',1),('B2+',2),('B3+',3)]:
  z=r[r.harm_n.le(1)&r.benefit_n.ge(minb)];ztp=int(z.new_true_print.sum());zp=z.loc[z.new_true_print.gt(0),'page'].nunique();zg=z.loc[z.new_true_print.gt(0),'source_group'].nunique()
  for x,y,v in itertools.product(D7_TP,D7_PAGES,D7_GROUPS):D7.append({'threshold':t,'benefit':label,'qualified_tp':ztp,'qualified_pages':zp,'qualified_groups':zg,'required_tp':x,'required_pages':y,'required_groups':v,'passes':ztp>=x and zp>=y and zg>=v and not p4.any()})
 for _,x in s.iterrows():
  rr=r[r.source_group.astype(str).eq(str(x.source_group))];q3=int(rr.harm_n.ge(3).sum());q4=int(rr.harm_n.eq(4).sum());D8.append({'threshold':t,'source_group':x.source_group,'pages':int(x.pages),'false_true_ratio':float(x.false_true_ratio),'utility':float(x.utility),'P3':q3,'P4':q4,'D8_R0':int(x.pages)<3 or (float(x.false_true_ratio)<=.25 and int(x.catastrophic_components)==0),'D8_R1':float(x.false_true_ratio)<=.35 and q3==0,'D8_R2':float(x.false_true_ratio)<=.50 and q4==0,'D8_R3_source_scoped_only':True})
per=pd.DataFrame(P);d8=pd.DataFrame(D8);util=pd.DataFrame(U);d7=pd.DataFrame(D7);front=per.assign(hard_stop=per.P4.gt(0)|per.repeated_P3,exploratory_only=True)
for d,n in [(per,'perceptual_D3_D8_results.csv'),(per,'audit_reserve_envelope_results.csv'),(d8,'D8_source_stability_variants.csv'),(util,'severity_adjusted_utility_matrix.csv'),(d7,'D7_sensitivity_matrix.csv'),(front,'envelope_relaxation_frontier.csv'),(pg,'results_by_page.csv'),(sg,'results_by_source_group.csv')]:csv(d,TABLES/n)
burden=pd.DataFrame([{'threshold':t,'review_units':len(active(t)),'P2_plus':int(active(t).harm_n.ge(2).sum())} for t in THRESHOLDS]);csv(burden,TABLES/'review_burden_summary.csv');csv(pd.DataFrame(),TABLES/'leave_one_group_out.csv')
qualified='single-reviewer-exploratory-with-provenance-warning' if manifest_summary['provenance_warning'] else 'single-reviewer-exploratory';verdict={'status':qualified,'production_authority':False,'threshold_adoption_authority':False,'direct_source_lock_sha256':source_lock_sha256,'notebook37_package_provenance_warning':manifest_summary['provenance_warning']};js(OUT/'notebook38_verdict.json',verdict);text(OUT/'executive_summary.txt',json.dumps(verdict,indent=2));text(OUT/'decision_record.md','# Notebook 38 Decision Record\n\n'+json.dumps(verdict,indent=2));print(verdict)


## 10. Visuals and final package


In [ ]:
plt.figure(figsize=(8,5));plt.plot(per.false_annotation,per.true_print,marker='o');plt.xlabel('False-annotation pixels');plt.ylabel('True-print pixels');plt.tight_layout();plt.savefig(VIS/'perceptual_frontier.png',dpi=160);plt.close();plt.figure(figsize=(8,5));plt.plot(per.threshold,per.P2_plus,marker='o');plt.xlabel('Threshold');plt.ylabel('P2+ units');plt.tight_layout();plt.savefig(VIS/'severity_by_threshold.png',dpi=160);plt.close()
items=[OUT/'protocol_lock.json',OUT/'severity_mapping_lock.json',OUT/'notebook38_verdict.json',OUT/'executive_summary.txt',OUT/'decision_record.md',TABLES/'review_unit_manifest.csv',TABLES/'review_split_map.csv',TABLES/'blinded_review_manifest.csv',R1,R2,TABLES/'reviewer_agreement.csv',TABLES/'registered_D3_D8_results.csv',TABLES/'perceptual_D3_D8_results.csv',TABLES/'audit_reserve_envelope_results.csv',TABLES/'envelope_relaxation_frontier.csv',TABLES/'results_by_page.csv',TABLES/'results_by_source_group.csv',TABLES/'leave_one_group_out.csv',TABLES/'review_burden_summary.csv',TABLES/'severity_adjusted_utility_matrix.csv',TABLES/'D7_sensitivity_matrix.csv',TABLES/'D8_source_stability_variants.csv',VIS/'perceptual_frontier.png',VIS/'severity_by_threshold.png',AUDIT/'notebook38_source_input_lock.csv',AUDIT/'nb37_manifest_verification.csv',AUDIT/'nb37_manifest_verification_summary.json',AUDIT/'registered_gate_replay.csv',AUDIT/'review_coverage.csv',AUDIT/'missing_review_assets.csv',REF/'notebook37_experiment_identity.json',REF/'source_v82_record.json']
missing=[str(x) for x in items if not x.exists()]
if missing:stop('package','Missing outputs',{'missing':missing})
UPLOAD.mkdir();rows=[]
for src in items:
 rel=src.relative_to(OUT);dst=UPLOAD/rel;dst.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(src,dst);rows.append({'relative_path':rel.as_posix(),'bytes':dst.stat().st_size,'sha256':sha(dst)})
text(UPLOAD/'README.txt','Single-reviewer exploratory package; no threshold or production authority.\n');rows.append({'relative_path':'README.txt','bytes':(UPLOAD/'README.txt').stat().st_size,'sha256':sha(UPLOAD/'README.txt')});csv(pd.DataFrame(rows),UPLOAD/'upload_manifest.csv');zpath=OUT/'notebook38_upload_bundle.zip'
with zipfile.ZipFile(zpath,'w',zipfile.ZIP_DEFLATED) as z:
 for p in UPLOAD.rglob('*'):
  if p.is_file():z.write(p,arcname=(Path('upload')/p.relative_to(UPLOAD)).as_posix())
js(OUT/'final_package_record.json',{'zip':str(zpath),'sha256':sha(zpath),'utc':now()});print(zpath)


## Interpretation boundary
Package-manifest discrepancies remain visible and qualify the exploratory result. Direct run-root inputs are independently fingerprinted. This notebook does not establish promotion-grade provenance and cannot nominate or adopt a relaxed envelope.
